# 🛢️ Ενότητα 9: Δοσοληψίες & Ασφάλεια (Transactions)

Μια βάση δεδομένων πρέπει να εξασφαλίζει την ακεραιότητα των δεδομένων, ειδικά όταν κάτι πάει στραβά (π.χ. διακοπή ρεύματος). Εδώ μπαίνουν οι Δοσοληψίες (Transactions) και οι ιδιότητες **ACID** (Atomicity, Consistency, Isolation, Durability).

In [ ]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE Accounts (ID INTEGER, Balance REAL)")
conn.executemany("INSERT INTO Accounts VALUES (?, ?)", [(1, 1000), (2, 500)])
conn.commit()

## COMMIT και ROLLBACK
Έστω ότι μεταφέρουμε 200€ από το λογαριασμό 1 στον λογαριασμό 2. Αυτό θέλει 2 κινήσεις (UPDATE). Αν η 2η αποτύχει, πρέπει να ακυρωθεί και η 1η! Αυτό γίνεται με τις δοσοληψίες.

Στην SQLite (και στην Python), ένα transaction ξεκινάει αυτόματα όταν κάνουμε αλλαγές και κλείνει με `conn.commit()`. Αν θέλουμε να αναιρέσουμε τις αλλαγές, καλούμε `conn.rollback()`.

In [ ]:
try:
    # Αφαιρούμε 200 από τον 1
    conn.execute("UPDATE Accounts SET Balance = Balance - 200 WHERE ID = 1;")
    
    # Κάτι πάει στραβά...
    raise Exception("Ωχ, έπεσε το σύστημα!")
    
    # Αυτό δεν θα τρέξει ποτέ
    conn.execute("UPDATE Accounts SET Balance = Balance + 200 WHERE ID = 2;")
    
    conn.commit()
except Exception as e:
    print(f"Σφάλμα: {e}")
    conn.rollback()
    print("Έγινε Rollback. Τα δεδομένα είναι ασφαλή.")

pd.read_sql_query("SELECT * FROM Accounts;", conn)